# 決算反省会サマリー

`PREDICT_DATE` を設定して全セル実行。

In [ ]:
PREDICT_DATE = "20260507"

In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import Markdown, display

os.environ["PYTHONUTF8"] = "1"

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / "scripts" / "earnings_model",
    Path("C:/Users/zonekun/Documents/codex/investment-agent/scripts/earnings_model"),
    Path("C:/gdrive/claude/investment-agent/scripts/earnings_model"),
]
for candidate in candidate_dirs:
    if (candidate / "download_review_data.py").exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError("download_review_data.py が見つかりません")

from download_review_data import LOCAL_DIR, download_for_date, get_gcs_client
from hanseikai_summary import generate_summary
from review_report import build_rows, load_json, write_csv, write_md

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: GCS DL
client = get_gcs_client()
download_for_date(client, PREDICT_DATE)

# Step 2: レポート生成
rows, counts = build_rows(PREDICT_DATE)
actual_meta = load_json(PREDICT_DATE, "actual")
csv_path = LOCAL_DIR / f"earnings_review_{PREDICT_DATE}.csv"
md_path = LOCAL_DIR / f"earnings_review_{PREDICT_DATE}.md"
write_csv(rows, csv_path)
write_md(rows, counts, PREDICT_DATE, actual_meta, md_path)

# Step 3: Markdown表示
summary = generate_summary(PREDICT_DATE, rows, counts, actual_meta)
summary_path = LOCAL_DIR / f"summary_{PREDICT_DATE}.md"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(summary + "\n")
display(Markdown(summary))